In [0]:
# Definir el esquema
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, LongType
schema = StructType([
    StructField("_c0", IntegerType(), True),           # índice
    StructField("trans_date_trans_time", StringType(), True),  # fecha y hora
    StructField("cc_num", LongType(), True),            # número de tarjeta
    StructField("merchant", StringType(), True),        # nombre del comercio
    StructField("category", StringType(), True),        # categoría
    StructField("amt", DoubleType(), True),             # monto
    StructField("first", StringType(), True),           # nombre
    StructField("last", StringType(), True),            # apellido
    StructField("gender", StringType(), True),          # género
    StructField("street", StringType(), True),          # calle
    StructField("city", StringType(), True),            # ciudad
    StructField("state", StringType(), True),           # estado
    StructField("zip", IntegerType(), True),            # código postal
    StructField("lat", DoubleType(), True),             # latitud titular
    StructField("long", DoubleType(), True),            # longitud titular
    StructField("city_pop", IntegerType(), True),       # población ciudad
    StructField("job", StringType(), True),             # ocupación
    StructField("dob", StringType(), True),             # fecha nacimiento
    StructField("trans_num", StringType(), True),       # código transacción
    StructField("unix_time", LongType(), True),         # tiempo unix
    StructField("merch_lat", DoubleType(), True),       # latitud comercio
    StructField("merch_long", DoubleType(), True),      # longitud comercio
    StructField("is_fraud", IntegerType(), True)        # 0=legítima, 1=fraude
])

In [0]:
# Leer con Try/Except
try:
    df = spark.read.format("csv") \
        .option("header", "true") \
        .schema(schema) \
        .load("/Volumes/fraud_project/bronze/raw_data/input/csv/fraudTest.csv")
    
    print("Archivo leído correctamente")
    print(f"Total de registros: {df.count()}")

except Exception as e:
    print(f"Error al leer el archivo: {str(e)}")

display(df)

In [0]:
df.write.format("parquet").mode("overwrite").save("/Volumes/fraud_project/bronze/raw_data/parquet")
df = spark.read.format("parquet").load("/Volumes/fraud_project/bronze/raw_data/parquet")
df.count()

In [0]:
print(df.columns)

In [0]:
%sql
-- Contar registros por categoría
SELECT category, count(1)
FROM fraud_project.bronze.transacciones_sin_particion
GROUP BY ALL

In [0]:
# Delta Table CON partición por category
df.write.format("delta").mode("overwrite").partitionBy("category").saveAsTable("fraud_project.bronze.transacciones_con_particion")

In [0]:
%sql
-- CON partición (más rápido)
SELECT * FROM fraud_project.bronze.transacciones_con_particion
WHERE category = 'food_dining'

In [0]:
# Delta Table SIN partición
df.write.format("delta").mode("overwrite").saveAsTable("fraud_project.bronze.transacciones_sin_particion")

In [0]:
%sql
-- SIN partición (más lento)
SELECT * FROM fraud_project.bronze.transacciones_sin_particion
WHERE category = 'food_dining'

In [0]:
%sql
-- Ver estado ANTES del cambio
SELECT is_fraud, count(1)
FROM fraud_project.bronze.transacciones_con_particion
WHERE category = 'food_dining'
GROUP BY ALL

In [0]:
%sql
-- Aplicar el UPDATE (operación de actualización)
UPDATE fraud_project.bronze.transacciones_con_particion
SET is_fraud = 1
WHERE category = 'food_dining'

In [0]:
%sql
-- Ver estado DESPUÉS del cambio
SELECT is_fraud, count(1)
FROM fraud_project.bronze.transacciones_con_particion
WHERE category = 'food_dining'
GROUP BY ALL

In [0]:
%sql
-- Ver el historial de versiones
DESCRIBE HISTORY fraud_project.bronze.transacciones_con_particion

In [0]:
%sql
-- Restaurar a la versión inicial
RESTORE TABLE fraud_project.bronze.transacciones_con_particion VERSION AS OF 0

In [0]:
%sql
-- Verificar que se restauró correctamente
SELECT is_fraud, count(1)
FROM fraud_project.bronze.transacciones_con_particion
WHERE category = 'food_dining'
GROUP BY ALL